# Lab 08-04 — Local vs global search over the GraphRAG index (HotpotQA)

**Track 08 · GraphRAG** — how the *same* graph index answers two very different ways.

An entity graph over a corpus can be queried two ways, and the choice is a real engineering tradeoff. Both strategies are drawn inline:

- **Local search** — `question → ChatOllama entity extraction → BGE cosine entity linking into the graph → 1-hop expansion → passages those entities live in → ChatOllama answer`. Precise — it only sees passages that touch the question — but it fails when the question's entities are not in the graph or the answer needs a *distant* part of the corpus.
- **Global search** — `question → BGE embedding → rank precomputed community summaries by cosine → top-2 summaries → ChatOllama answer`. It ignores the graph topology for retrieval and sees the whole corpus (compressed into summaries), so it can answer corpus-level questions, but it is lossier — the answer must survive the map/reduce compression.

This notebook is **self-contained**: it imports `langchain-ollama`, `langchain-huggingface`, `networkx`, and `scikit-network` directly — no repo component library. Every block of the pipeline is built right here: the local-LLM wrapper, the triple/entity extractors, the graph builder, the Leiden community detector, the map summarizer, the BGE embedder, and both search routines all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

The lab runs both strategies on the same HotpotQA questions. Each question carries 10 candidate paragraphs; we build a graph over them, detect communities, summarize the top ones, then evaluate both searches against the dataset's gold answer and gold paragraphs (`supporting_facts`).


## Setup

Two local prerequisites must hold — there are **no API calls** in this lab, so it costs zero quota:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — the local LLM behind extraction, summarization, and both answer calls (talked to through `langchain-ollama`). If the server is not up, nothing that touches the LLM can run.
- **hotpotqa on disk** — `Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json`, already fetched by the repo's manifest-verified fetchers.

Embeddings are local too: `HuggingFaceEmbeddings` with `BAAI/bge-base-en-v1.5` on CPU (`normalize_embeddings=True`, which BGE requires for cosine) — the same behavior as the repo's BGE block. No repo imports are needed: everything comes from `langchain-ollama`, `langchain-huggingface`, `networkx`, `scikit-network`, `numpy`, and `scipy`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no sys.path trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`):
#   langchain-ollama      -> ChatOllama, the local Ollama chat backend
#   langchain-huggingface -> HuggingFaceEmbeddings (local BGE, CPU)
#   sentence-transformers -> the embedding model behind HuggingFaceEmbeddings
#   networkx              -> the entity graph + Louvain fallback
#   scikit-network        -> the Leiden community detector (pulls numpy/scipy)
%pip install -q langchain-ollama langchain-huggingface sentence-transformers networkx scikit-network


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import time
from pathlib import Path

# networkx + langchain-ollama + langchain-huggingface — the only libraries
# this notebook needs at import time. Nothing is imported from the repo's
# src/ component library (numpy/scipy/sknetwork are pulled lazily inside
# the community detector, exactly like the shared tool does).
import networkx as nx
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_QUESTIONS = 3` bounds the run — each question costs roughly 16 LLM calls (10 passage extractions, 1 entity extraction, up to 4 community summaries, 1 local answer, 1 global answer), so the question count *is* the quota knob. `MAX_COMMUNITIES = 4` caps the per-question map step; `LINK_TOP_N = 6` bounds how many passages local search may surface; `LINK_THRESHOLD = 0.55` is the minimum cosine for an entity link. `BGE_MODEL_NAME` selects the local embedder and `BGE_DEVICE = "cpu"` keeps VRAM for Ollama.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
HOTPOT_PATH = Path("Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json")
N_QUESTIONS = 3  # each question costs ~16 LLM calls; keep the lab fast
MAX_COMMUNITIES = 4  # per-question cap on community summaries
LINK_TOP_N = 6  # max passages local search may surface
LINK_THRESHOLD = 0.55  # minimum cosine for entity linking
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM


## 2. Load — first N HotpotQA questions (each has 10 context paragraphs)

`hotpot_dev_distractor_v1.json` is a list of records, each carrying a `question`, a `context` of 10 candidate paragraphs (title + sentences — the distractors a retriever must cut through), an `answer`, and `supporting_facts` — the gold `(title, sentence_index)` evidence pairs. `load_questions` keeps the first `n` records (deterministic head); `question_passages` flattens each context paragraph's sentences into one text plus its title; `gold_paragraphs` extracts the required paragraph titles; and `answer_contains` is the normalized gold-in-answer substring check both searches are scored with.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N HotpotQA questions (each has 10 context paragraphs)
# --------------------------------------------------------------------------
def load_questions(path: Path, n: int) -> list[dict]:
    """Return the first ``n`` questions from the dev set."""
    with open(path) as f:
        records = json.load(f)
    return records[:n]


def question_passages(rec: dict) -> tuple[list[str], list[str]]:
    """Return (passage_texts, paragraph_titles) for the 10 context paragraphs."""
    texts: list[str] = []
    titles: list[str] = []
    for title, sentences in rec["context"]:
        titles.append(title)
        texts.append(" ".join(sentences))
    return texts, titles


def gold_paragraphs(rec: dict) -> set[str]:
    """The paragraph titles HotpotQA marks as required evidence."""
    return {title for title, _ in rec["supporting_facts"]}


def answer_contains(gold: str, answer: str) -> bool:
    """Normalized substring check: is the gold answer inside the answer?"""
    return gold.strip().lower() in answer.strip().lower()


## 3. Experiment — build a per-question index, run both searches, evaluate

Everything from labs 01 and 03 is rebuilt inline here, plus the two query strategies:

- **Index** — `_OllamaLLM` (ChatOllama with code-fence stripping and JSON retries), `extract_triples` / `extract_entities` (the JSON extraction and entity-listing prompts), `build_entity_graph` (networkx fold), `detect_communities` (sknetwork Leiden, networkx Louvain fallback, isolated nodes recovered as singletons), and `community_summaries` (map step: community relations rendered as lines, then condensed by the LLM).
- **Local** — `link_entities` embeds the question's extracted entities and every graph node with the local BGE embedder and best-matches each query entity to a graph node above `LINK_THRESHOLD` cosine (the lexical-alias bridge: the question says "Scott Derrickson" and the graph node is "Scott Derrickson"); the linked nodes and their 1-hop neighbors contribute passages, capped at `LINK_TOP_N`, and the LLM answers from that context.
- **Global** — the question vector is scored against every community-summary vector, the top 2 summaries become the context, and the LLM answers from them.

Each question is evaluated three ways: does the local answer contain the gold, does the global answer contain the gold, and did local search surface a gold paragraph. `run_experiment` returns per-question rows plus 0/1 aggregates.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — build a per-question index, run both searches, evaluate
# --------------------------------------------------------------------------
# Inline replacement for the repo's OllamaLLM adapter (src/llms/ollama.py):
# the same invoke / json_object contract the lab's extractor relies on.
TRIPLE_SCHEMA = '{"triples": [{"head": "...", "relation": "...", "tail": "..."}]}'


class _OllamaLLM:
    """Generate / extract with the locally served Ollama model.

    Contract mirror of the repo's OllamaLLM adapter, built inline so this
    notebook needs no repo imports: ``invoke`` returns the chat text,
    ``json_object`` strips a surrounding markdown code fence and re-prompts
    on parse failure.
    """

    def __init__(self, model: str = "qwen2.5-coder:7b", temperature: float = 0.0,
                 base_url: str = "http://localhost:11434"):
        self.model = model
        self.temperature = temperature
        self.base_url = base_url
        self._llm = None

    def _get_llm(self) -> ChatOllama:
        if self._llm is None:
            self._llm = ChatOllama(
                model=self.model, temperature=self.temperature, base_url=self.base_url
            )
        return self._llm

    def invoke(self, prompt: str) -> str:
        return self._get_llm().invoke(prompt).content

    @staticmethod
    def _strip_code_fence(text: str) -> str:
        """Remove a surrounding markdown code fence (```json ... ```)."""
        lines = text.strip().splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        return "\n".join(lines).strip()

    def json_object(self, prompt: str, retries: int = 2) -> dict:
        """Ask the model to output ONLY a JSON object and parse it."""
        text = ""
        for attempt in range(retries + 1):
            full = prompt if attempt == 0 else prompt + (
                "\n\nRespond with ONLY valid JSON, no markdown.")
            text = self.invoke(full)
            try:
                parsed = json.loads(self._strip_code_fence(text))
                if isinstance(parsed, (dict, list)):
                    return parsed
            except (json.JSONDecodeError, ValueError):
                pass
        return {"error": f"could not parse JSON after {retries + 1} attempts",
                "raw": text}


def extract_triples(llm, text: str) -> list[tuple[str, str, str]]:
    """Extract ``(head, relation, tail)`` triples from one passage of text."""
    prompt = (
        "Extract the entity-relation triples from the text below.\n"
        "Rules:\n"
        "- Only entities explicitly named in the text.\n"
        "- Entities are people, places, organizations, works, events or "
        "concrete things (1-4 words).\n"
        "- Relations are short verbs or prepositional phrases (1-4 words), "
        "present tense.\n"
        f"- Output ONLY JSON: {TRIPLE_SCHEMA}\n"
        "- Output an empty list if the text has no meaningful triples.\n"
        "\n"
        f"Text:\n{text}"
    )
    result = llm.json_object(prompt)
    if isinstance(result, list):  # bare array of triples, no wrapper key
        raw = result
    elif isinstance(result, dict) and "error" not in result:
        raw = result.get("triples", result.get("edges", result.get("data", [])))
        if isinstance(raw, dict):  # single triple given without a list wrapper
            raw = [raw]
    else:
        return []
    triples: list[tuple[str, str, str]] = []
    for item in raw or []:
        if not isinstance(item, dict):
            continue
        head = str(item.get("head", item.get("subject", ""))).strip()
        relation = str(item.get("relation", item.get("predicate", ""))).strip()
        tail = str(item.get("tail", item.get("object", ""))).strip()
        if head and tail:
            triples.append((head, relation or "related to", tail))
    return triples


def extract_entities(llm, text: str, limit: int = 8) -> list[str]:
    """Ask the LLM for the salient named entities in ``text`` (up to ``limit``)."""
    prompt = (
        "List the salient named entities in the text below (people, places, "
        "organizations, works, events, concrete things).\n"
        "Rules:\n"
        "- Only entities explicitly named in the text.\n"
        "- Each entity is 1-4 words; use the most specific name that appears.\n"
        f"- Output ONLY JSON: {{\"entities\": [\"...\", \"...\"]}} with at most "
        f"{limit} entities.\n"
        "\n"
        f"Text:\n{text}"
    )
    result = llm.json_object(prompt)
    if isinstance(result, list):  # bare array of entity names, no wrapper key
        entities = result
    elif isinstance(result, dict) and "error" not in result:
        entities = result.get("entities", [])
    else:
        return []
    out: list[str] = []
    for ent in entities:
        if isinstance(ent, str) and ent.strip():
            out.append(ent.strip())
    return out[:limit]


def build_entity_graph(passages: list[str], llm, progress=None) -> nx.Graph:
    """Fold every passage's triples into one networkx entity graph."""
    graph = nx.Graph()
    total = len(passages)
    for i, text in enumerate(passages):
        for head, relation, tail in extract_triples(llm, text):
            if head == tail:
                continue  # self-loops carry no structure
            graph.add_edge(head, tail, relations=set())
            for node in (head, tail):
                graph.nodes[node].setdefault("passages", [])
            graph.nodes[head]["passages"].append(i)
            graph.nodes[tail]["passages"].append(i)
            graph[head][tail]["relations"].add(relation)
        if progress is not None:
            progress(i + 1, total)
    for _, _, data in graph.edges(data=True):
        data["weight"] = len(data["relations"])
    return graph


def detect_communities(graph: nx.Graph, seed: int = 42) -> list[set[str]]:
    """Partition ``graph`` into communities (Leiden, Louvain fallback)."""
    nodes = list(graph.nodes())
    if not nodes:
        return []
    try:
        import numpy as np
        from scipy.sparse import csr_matrix
        from sknetwork.clustering import Leiden

        index = {node: i for i, node in enumerate(nodes)}
        n = len(nodes)
        rows: list[int] = []
        cols: list[int] = []
        for u, v in graph.edges():
            rows.extend((index[u], index[v]))
            cols.extend((index[v], index[u]))
        adjacency = csr_matrix(
            (np.ones(len(rows)), (rows, cols)), shape=(n, n)
        )
        membership = Leiden(random_state=seed).fit_transform(adjacency)
        dense = membership.toarray()
        labels = dense.argmax(axis=1)
        assigned = dense.sum(axis=1) > 0
    except Exception:
        # Fallback: networkx Louvain (unweighted, to match the ones/zeros above)
        parts = nx.community.louvain_communities(
            graph, weight=None, seed=seed
        )
        return [set(part) for part in parts]

    communities: dict[int, set[str]] = {}
    next_label = dense.shape[1]
    for node, label, is_assigned in zip(nodes, labels, assigned):
        if is_assigned:
            communities.setdefault(int(label), set()).add(node)
        else:
            communities[next_label] = {node}  # isolated -> own community
            next_label += 1
    return list(communities.values())


def community_text(graph: nx.Graph, community) -> str:
    """Render the intra-community edges of ``community`` as relation lines."""
    members = set(community)
    lines: set[str] = set()
    for node in members:
        for neighbor in graph.neighbors(node):
            if neighbor not in members:
                continue
            for relation in graph[node][neighbor].get("relations", ()):
                lines.add(f"{node} -[{relation}]-> {neighbor}")
    return "\n".join(sorted(lines)) or "no intra-community relations"


def summarize_community(llm, text: str) -> str:
    """Map step: condense one community's relation lines into prose."""
    prompt = (
        "Summarize the following entity-relation fragment in 2-3 sentences. "
        "Name the main entities and what connects them.\n\n"
        f"{text}"
    )
    return llm.invoke(prompt).strip()


def community_summaries(llm, graph: nx.Graph, communities, max_communities: int = 8) -> list[dict]:
    """Map/reduce-ready list of ``{"members": [...], "summary": "..."}``."""
    ordered = sorted(communities, key=len, reverse=True)
    summaries: list[dict] = []
    for community in ordered[:max_communities]:
        summaries.append(
            {
                "members": sorted(community),
                "size": len(community),
                "summary": summarize_community(
                    llm, community_text(graph, community)
                ),
            }
        )
    return summaries


def _cosine(a, b) -> float:
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(x * x for x in b) ** 0.5
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return sum(x * y for x, y in zip(a, b)) / (norm_a * norm_b)


def link_entities(embedder, query_entities, candidates, threshold: float = 0.55) -> dict[str, str]:
    """Best-match each query entity to a graph entity via cosine similarity."""
    if not query_entities or not candidates:
        return {}
    query_vecs = embedder.embed_documents(list(query_entities))
    candidate_vecs = embedder.embed_documents(list(candidates))
    linked: dict[str, str] = {}
    for query, query_vec in zip(query_entities, query_vecs):
        best_name, best_score = None, threshold
        for name, vec in zip(candidates, candidate_vecs):
            score = _cosine(query_vec, vec)
            if score > best_score:
                best_name, best_score = name, score
        if best_name is not None:
            linked[query] = best_name
    return linked


def local_search(question, llm, embedder, graph, passages,
                 top_n: int = 6, threshold: float = 0.55) -> dict:
    """Local search: link question entities into the graph, expand 1 hop."""
    query_entities = extract_entities(llm, question)
    linked = link_entities(embedder, query_entities, graph.nodes(), threshold)
    seed = set(linked.values())
    expanded = set(seed)
    for node in seed:
        expanded.update(graph.neighbors(node))

    passage_ids: set[int] = set()
    for node in expanded:
        passage_ids.update(graph.nodes[node].get("passages", ()))
    ordered_ids = sorted(passage_ids)[:top_n]
    retrieved = [passages[i] for i in ordered_ids]

    context = "\n\n".join(f"[{i}] {text}" for i, text in enumerate(retrieved))
    answer = llm.invoke(
        "Answer the question using ONLY the context paragraphs below. "
        "If the context does not contain the answer, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer in one or two sentences."
    ).strip()
    return {
        "answer": answer,
        "retrieved": retrieved,
        "retrieved_ids": ordered_ids,
        "query_entities": query_entities,
        "linked": linked,
        "expanded": sorted(expanded),
    }


def global_search(question, llm, embedder, community_summaries, top_n: int = 2) -> dict:
    """Global search: route the question to the most relevant summaries."""
    if not community_summaries:
        return {"answer": "", "summaries_used": [], "scores": []}
    question_vec = embedder.embed_documents([question])[0]
    summary_vecs = embedder.embed_documents(
        [entry["summary"] for entry in community_summaries]
    )
    scored = [
        (_cosine(question_vec, vec), entry)
        for vec, entry in zip(summary_vecs, community_summaries)
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    used = [entry for _, entry in scored[:top_n]]

    bullets = "\n".join(f"- {entry['summary']}" for entry in used)
    answer = llm.invoke(
        "Answer the question using ONLY the corpus summaries below. "
        "If the summaries do not contain the answer, say so.\n\n"
        f"Corpus summaries:\n{bullets}\n\n"
        f"Question: {question}\n\n"
        "Answer in one or two sentences."
    ).strip()
    return {
        "answer": answer,
        "summaries_used": [entry["summary"] for entry in used],
        "scores": [
            {"summary": entry["summary"], "cosine": round(score, 3)}
            for score, entry in scored[:top_n]
        ],
    }


def run_one_question(rec: dict, llm, embedder) -> dict:
    passages, titles = question_passages(rec)
    gold = rec["answer"]
    gold_titles = gold_paragraphs(rec)

    graph = build_entity_graph(passages, llm)
    communities = detect_communities(graph, seed=42)
    summaries = community_summaries(
        llm, graph, communities, max_communities=MAX_COMMUNITIES
    )

    local = local_search(
        rec["question"], llm, embedder, graph, passages,
        top_n=LINK_TOP_N, threshold=LINK_THRESHOLD,
    )
    glob = global_search(rec["question"], llm, embedder, summaries)

    local_gold_ids = [
        titles[i] for i in local["retrieved_ids"]
        if titles[i] in gold_titles
    ]
    return {
        "question": rec["question"],
        "gold": gold,
        "gold_titles": sorted(gold_titles),
        "local": {
            "answer": local["answer"],
            "linked": local["linked"],
            "gold_titles_retrieved": local_gold_ids,
        },
        "global": {
            "answer": glob["answer"],
        },
        "scores": {
            "local_answer_contains_gold": answer_contains(gold, local["answer"]),
            "global_answer_contains_gold": answer_contains(gold, glob["answer"]),
            "local_gold_para_recall": len(local_gold_ids) > 0,
        },
    }


def run_experiment() -> dict:
    questions = load_questions(HOTPOT_PATH, N_QUESTIONS)
    llm = _OllamaLLM()
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": BGE_DEVICE},
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )

    t0 = time.perf_counter()
    rows = [run_one_question(rec, llm, embedder) for rec in questions]
    total_s = time.perf_counter() - t0

    return {
        "rows": rows,
        "total_s": total_s,
        "agg": {
            "local_answer_contains_gold": sum(
                r["scores"]["local_answer_contains_gold"] for r in rows
            ),
            "global_answer_contains_gold": sum(
                r["scores"]["global_answer_contains_gold"] for r in rows
            ),
            "local_gold_para_recall": sum(
                r["scores"]["local_gold_para_recall"] for r in rows
            ),
            "questions": len(rows),
        },
    }


## 4. Demo — print the artifact

The demo prints, per question: the gold answer and gold paragraphs, which entities local search linked into the graph, both hit flags, and both answers (truncated). Then the aggregates — how many of the questions each strategy answered (gold contained) and whether local search surfaced a gold paragraph — and the takeaway on the precision/recall tradeoff between the two search modes.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 08-04 — Local vs global search over the GraphRAG index")
    print(f"{exp['agg']['questions']} questions in {exp['total_s']:.1f}s")
    print("=" * 66)

    for i, row in enumerate(exp["rows"], start=1):
        print(f"\nQ{i}: {row['question'][:90]}")
        print(f"    gold answer : {row['gold']}")
        print(f"    gold paras  : {', '.join(row['gold_titles'])[:80]}")
        print(f"    linked      : {row['local']['linked']}")
        print(f"    local  hit  : {row['scores']['local_answer_contains_gold']} "
              f"(gold paras {row['local']['gold_titles_retrieved']})")
        print(f"    global hit  : {row['scores']['global_answer_contains_gold']}")
        print(f"    local  ans  : {row['local']['answer'][:110]}")
        print(f"    global ans  : {row['global']['answer'][:110]}")

    a = exp["agg"]
    print(f"\n[5] Aggregates over {a['questions']} questions")
    print(f"    local  answer contains gold : {a['local_answer_contains_gold']}/{a['questions']}")
    print(f"    global answer contains gold : {a['global_answer_contains_gold']}/{a['questions']}")
    print(f"    local  surfaced a gold para : {a['local_gold_para_recall']}/{a['questions']}")

    print(f"\n[6] Takeaway")
    print("    Local search is entity-precise but narrow: it only sees")
    print("    passages that touch the question's entities, so it shines on")
    print("    multi-hop questions whose bridge entities are in the graph.")
    print("    Global search trades that precision for recall: the question")
    print("    is answered from compressed community summaries, so it can")
    print("    generalize but may lose exact facts during map/reduce.")


## 5. Verification gate

The lab ships a `--verify` gate — the same gate `python src/curriculum/08-graphrag/04-local-global.py --verify` runs: exactly `N_QUESTIONS` processed, local search linked entities for at least one question, every answer non-empty, local search surfaced a gold paragraph for at least one question, and all per-question scores are clean 0/1 flags. The gate turns "the lab ran" into "the lab ran *correctly*" — and its checks are deliberately tolerant of LLM variance, because answer quality on 3 questions is a demo, not a benchmark.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    a = exp["agg"]

    checks.append((f"exactly {N_QUESTIONS} questions processed",
                   a["questions"] == N_QUESTIONS))
    checks.append(("local search linked entities for >= 1 question",
                   any(r["local"]["linked"] for r in exp["rows"])))
    checks.append(("every answer is non-empty",
                   all(r["local"]["answer"].strip()
                       for r in exp["rows"])
                   and all(r["global"]["answer"].strip()
                           for r in exp["rows"])))
    checks.append(("local search surfaced a gold paragraph for >= 1 question",
                   a["local_gold_para_recall"] >= 1))
    checks.append(("per-question scores are 0/1 flags (reportable)",
                   all(v in (0, 1)
                       for r in exp["rows"]
                       for v in r["scores"].values())))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

This is the slow cell: per question roughly 10 extraction calls (one per context paragraph), 1 entity extraction, up to 4 community summaries, 1 local answer and 1 global answer — against a local `qwen2.5-coder:7b`, plus a BGE embedding pass on CPU for linking and summary routing. Expect several minutes for the 3 questions; no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

Per-question hit flags and answers for both strategies, plus the aggregates: how often each search mode contained the gold answer, and how often local search surfaced a gold paragraph.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check that Ollama is serving `qwen2.5-coder:7b`, that the hotpotqa json is intact, and that BGE can load on the configured device.


In [ ]:
verify_gate(exp)
